In [ ]:
import os
from google.colab import files, drive

drive.mount('/content/drive')

if not os.path.exists('/root/.kaggle/kaggle.json'):
    files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d ejlok1/cremad
!mkdir -p /content/cremad_raw
!unzip -q cremad.zip -d /content/cremad_raw
print("Dataset Ready!")

Mounted at /content/drive


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/ejlok1/cremad
License(s): ODC Attribution License (ODC-By)
100% 451M/451M [00:07<00:00, 66.2MB/s]

Dataset Ready!


In [ ]:
import librosa
import numpy as np
from tqdm import tqdm

def extract_augmented_5ch(file_path, augment=False, target_frames=94):
    # Loading 3 seconds of audio
    y, sr = librosa.load(file_path, duration=3.0, sr=22050)

    if len(y) < sr * 3:
        y = np.pad(y, (0, sr * 3 - len(y)))
    else:
        y = y[:sr * 3]
    if augment:
        if np.random.random() > 0.5:
            y = -y
        noise = np.random.randn(len(y))
        y = y + 0.005 * noise

    # 5 Channel Feature extraction
    #Mel Spectrogram (128 bins)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    ch1 = librosa.power_to_db(mel, ref=np.max)

    #MFCC (128 coefficients)
    ch2 = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=128)

    #Chroma STFT (Padded to 128 height)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr, n_chroma=12)
    ch3 = np.tile(chroma, (11, 1))[:128, :]

    #Spectral Contrast (Padded to 128 height)
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr, n_bands=6)
    ch4 = np.tile(contrast, (19, 1))[:128, :]

    #Tonnetz (Padded to 128 height)
    tonnetz = librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=sr)
    ch5 = np.tile(tonnetz, (22, 1))[:128, :]

    # Stack into (Height: 128, Time: 94, Channels: 5)
    stack = np.stack([ch1, ch2, ch3, ch4, ch5], axis=-1)
    return stack[:, :target_frames, :].astype(np.float32)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_60plus_model(input_shape=(128, 94, 5)):
    inputs = layers.Input(shape=input_shape)

    # CNN PATH (Texture & Frequency)
    cnn = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    cnn = layers.BatchNormalization()(cnn)
    cnn = layers.MaxPooling2D((2, 2))(cnn)

    cnn = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(cnn)
    cnn = layers.BatchNormalization()(cnn)
    cnn = layers.GlobalAveragePooling2D()(cnn)

    temporal = layers.Reshape((94, -1))(inputs)
    temporal = layers.Bidirectional(layers.LSTM(64, return_sequences=False))(temporal)

    #FUSION
    combined = layers.concatenate([cnn, temporal])

    x = layers.Dense(256, activation='swish')(combined)
    x = layers.Dropout(0.4)(x)
    x = layers.BatchNormalization()(x)

    outputs = layers.Dense(6, activation='softmax')(x) # 6 Emotions in CREMA-D

    model = models.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model = build_60plus_model()
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 94,   │          0 │ -                 │
│ (InputLayer)        │ 5)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 128, 94,   │      2,944 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 128, 94,   │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 64, 47,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 64, 47,    │     73,856 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 47,    │        512 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 94, 640)   │          0 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ batch_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 128)       │    360,960 │ reshape[0][0]     │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 256)       │          0 │ global_average_p… │
│ (Concatenate)       │                   │            │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │     65,792 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dropout[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 6)         │      1,542 │ batch_normalizat… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 506,886 (1.93 MB)

 Trainable params: 505,990 (1.93 MB)

 Non-trainable params: 896 (3.50 KB)

In [ ]:
import pandas as pd
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

# CREMA-D Emotion mapping
emotion_map = {
    'ANG': 'angry',
    'DIS': 'disgust',
    'FEA': 'fear',
    'HAP': 'happy',
    'NEU': 'neutral',
    'SAD': 'sad'
}

data_path = "/content/cremad_raw"

file_paths = []
labels = []

for root, dirs, files_list in os.walk(data_path):
    for file in files_list:
        if file.endswith('.wav'):
            part = file.split('_')
            if len(part) >= 3:
                emotion_code = part[2]
                if emotion_code in emotion_map:
                    file_paths.append(os.path.join(root, file))
                    labels.append(emotion_map[emotion_code])

if len(file_paths) == 0:
    print("CRITICAL ERROR: No .wav files found! Check your unzip path.")
    print(f"Current contents of {data_path}: {os.listdir(data_path)}")
else:
    lb = LabelEncoder()
    labels_encoded = to_categorical(lb.fit_transform(labels))

    print(f"Success! Total files found: {len(file_paths)}")
    print(f"Classes found: {lb.classes_}")
    print(f"Sample path: {file_paths[0]}")

Success! Total files found: 7442
Classes found: ['angry' 'disgust' 'fear' 'happy' 'neutral' 'sad']
Sample path: /content/cremad_raw/AudioWAV/1047_IEO_NEU_XX.wav


In [ ]:
X_train_paths, X_val_paths, y_train, y_val = train_test_split(
    file_paths, labels_encoded, test_size=0.2, random_state=42, stratify=labels_encoded
)

def process_set(paths, augment=False):
    features = []
    for p in tqdm(paths):
        feat = extract_augmented_5ch(p, augment=augment)
        features.append(feat)
    return np.array(features)

print("Extracting Training Features (with augmentation)...")
X_train = process_set(X_train_paths, augment=True)

print("Extracting Validation Features...")
X_val = process_set(X_val_paths, augment=False)

print(f"Training shape: {X_train.shape}")

Extracting Training Features (with augmentation)...


100%|██████████| 5953/5953 [45:20<00:00,  2.19it/s]


Extracting Validation Features...


100%|██████████| 1489/1489 [09:59<00:00,  2.48it/s]


Training shape: (5953, 128, 94, 5)


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

checkpoint = ModelCheckpoint('best_model.keras', monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

Epoch 1/100
187/187 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.2969 - loss: 1.9062
Epoch 1: val_accuracy improved from None to 0.17058, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras
187/187 ━━━━━━━━━━━━━━━━━━━━ 555s 3s/step - accuracy: 0.3272 - loss: 1.7536 - val_accuracy: 0.1706 - val_loss: 6.5982 - learning_rate: 0.0010
Epoch 2/100
187/187 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.3451 - loss: 1.6164
Epoch 2: val_accuracy did not improve from 0.17058
187/187 ━━━━━━━━━━━━━━━━━━━━ 555s 3s/step - accuracy: 0.3632 - loss: 1.5725 - val_accuracy: 0.1706 - val_loss: 8.6143 - learning_rate: 0.0010
Epoch 3/100
187/187 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.4091 - loss: 1.4784
Epoch 3: val_accuracy improved from 0.17058 to 0.17730, saving model to best_model.keras

Epoch 3: finished saving model to best_model.keras
187/187 ━━━━━━━━━━━━━━━━━━━━ 565s 3s/step - accuracy: 0.4111 - loss: 1.4679 - val_accuracy: 0.1773 - val_loss: 7.1769 - learning